<a href="https://colab.research.google.com/github/reddybvr/AIML-Agentic_Learning/blob/build_agenticai/Ahmed_LLM_as_Judge_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧑‍⚖️ Build a LLM Judge for Translation Evaluation

This Colab notebook guides you through building a **LLM-based evaluator** (or "judge") for translated sentences. The judge will score translations on:

- **Accuracy** (semantic fidelity)
- **Understandability** (clarity and fluency)
- **Hallucination** (whether extra or missing content exists)

We'll also explore **back-translation** as a consistency check.

## 🛠️ Step 1: Install and Import Dependencies

In [ ]:
!pip install --quiet openai tiktoken tqdm

%pip install openai tqdm
from openai import OpenAI

In [ ]:
import os
from tqdm import tqdm
import getpass
from getpass import getpass
from google.colab import userdata
import os
from openai import OpenAI

## 🔐 Step 2: Set Your OpenAI API Key

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

Enter your OpenAI API key: ··········


In [ ]:
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

## 📥 Step 3: Prepare a Sample Dataset

In [ ]:
sample_data = [
    {
        "source_text": "We need to finalize the contract before the end of the month.",
        "translated_text": "月末までに契約を確定させる必要があります。" # The contract needs to be finalized by the end of the month.
    },
    {
        "source_text": "Schedule a meeting with the design team for next Tuesday.",
        "translated_text": "来週の日曜日にデザインチームとの会議を予定してください。" # Please schedule a meeting with the design team next Sunday.
    },
    {
        "source_text": "All passwords must be changed every 90 days.",
        "translated_text": "すべてのパスワードは91日ごとに変更する必要があります。" # All passwords must be changed every 91 days.
    }
]

## 🤖 Step 4: Define the LLM Judge Prompt with Few-Shot Examples

In [ ]:

def build_judge_prompt(source_text, translated_text):
    few_shot_examples = """Example 1:
Original: Please send the invoice by Friday.
Translation: 金曜日までに請求書を送ってください。

Evaluation:
{
  "accuracy_score": 5,
  "accuracy_notes": "All details, including deadline and intent, are preserved.",
  "understandability_score": 5,
  "understandability_notes": "Fluent and idiomatic phrasing.",
  "hallucination_score": 5,
  "hallucination_notes": "No content was added or omitted."
}

---

Example 2:
Original: Transfer $2000 to Mike by noon.
Translation: マイクに2000ドルを送ってください。

Evaluation:
{
  "accuracy_score": 3,
  "accuracy_notes": "Correct recipient and amount, but deadline 'by noon' is missing.",
  "understandability_score": 5,
  "understandability_notes": "Clear and fluent phrasing.",
  "hallucination_score": 4,
  "hallucination_notes": "Omission of deadline makes it slightly inaccurate."
}

---

Example 3:
Original: Do not share this file with anyone outside the company.
Translation: このファイルを誰とも共有しないでください。

Evaluation:
{
  "accuracy_score": 4,
  "accuracy_notes": "Prohibition is correctly translated, but lacks explicit mention of 'outside the company'.",
  "understandability_score": 5,
  "understandability_notes": "Fluent and natural Japanese.",
  "hallucination_score": 4,
  "hallucination_notes": "Missing detail about 'outside the company'."
}

---"""

    task = f"""
Act as an impartial judge and evaluate the qaulity of the translation provided
by an LLM

First here are some example evaluations:

{few_shot_examples}

Now, evaluate the following translation:

Original: {source_text}
Translation: {translated_text}

Respond in the following JSON format:

{{
  "accuracy_score": ...,
  "accuracy_notes": "...",
  "understandability_score": ...,
  "understandability_notes": "...",
  "hallucination_score": ...,
  "hallucination_notes": "..."
}}
"""
    return task


## 🧪 Step 5: Run the Judge on Sample Data

In [ ]:
# sample_data = [
#     {
#         "source_text": "We need to finalize the contract before the end of the month.",
#         "translated_text": "月末までに契約を確定させる必要があります。" # The contract needs to be finalized by the end of the month.
#     },
#     {
#         "source_text": "Schedule a meeting with the design team for next Tuesday.",
#         "translated_text": "来週の日曜日にデザインチームとの会議を予定してください。" # Please schedule a meeting with the design team next Sunday.
#     },
#     {
#         "source_text": "All passwords must be changed every 90 days.",
#         "translated_text": "すべてのパスワードは91日ごとに変更する必要があります。" # All passwords must be changed every 91 days.
#     }
# ]

In [ ]:
def evaluate_with_llm(sample_data):
    for item in sample_data:
        prompt = build_judge_prompt(item["source_text"], item["translated_text"])
        print("🔍 Evaluating:")
        print(f"Original: {item['source_text']}")
        print(f"Translation: {item['translated_text']}")
        print("🧠 GPT Response:")

        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a strict bilingual translation evaluator."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        resp = response.choices[0].message.content
        print("LLM output is ", type(resp))
        print(resp)


## ✅ Step 6: Try It Out

In [ ]:
evaluate_with_llm(sample_data)

🔍 Evaluating:
Original: We need to finalize the contract before the end of the month.
Translation: 月末までに契約を確定させる必要があります。
🧠 GPT Response:
LLM output is  <class 'str'>
{
  "accuracy_score": 5,
  "accuracy_notes": "All details, including the deadline and the action to be taken, are accurately translated.",
  "understandability_score": 5,
  "understandability_notes": "The translation is fluent and idiomatic, easy to understand for a native speaker.",
  "hallucination_score": 5,
  "hallucination_notes": "No content was added or omitted in the translation."
}
🔍 Evaluating:
Original: Schedule a meeting with the design team for next Tuesday.
Translation: 来週の日曜日にデザインチームとの会議を予定してください。
🧠 GPT Response:
LLM output is  <class 'str'>
{
  "accuracy_score": 3,
  "accuracy_notes": "The day of the week is incorrectly translated as Sunday instead of Tuesday.",
  "understandability_score": 5,
  "understandability_notes": "The sentence is grammatically correct and understandable, but the day of the week is 

In [ ]:
import json

In [ ]:
resp = """
{
  "accuracy_score": 5,
  "accuracy_notes": "All details, including the deadline and the action to be taken, are accurately translated.",
  "understandability_score": 5,
  "understandability_notes": "The sentence is fluent and idiomatic in Japanese.",
  "hallucination_score": 5,
  "hallucination_notes": "No content was added or omitted in the translation."
}
"""

d = json.loads(resp)

print(type(d))

<class 'dict'>


In [ ]:
d["accuracy_score"]

5

## 🔁 Step 7 (Optional): Add Back-Translation Support

In [ ]:
def get_back_translation(translated_text, target_lang="Japanese"):
    back_prompt = f"Translate this {target_lang} sentence back into English:\n\n{translated_text}"
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": back_prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


In [ ]:
def build_judge_prompt_with_back_translation(source_text, translated_text, back_translated_text):
    return f"""Evaluate the quality of this translation using both the original source and the back-translation.

Source (English): {source_text}
Translation (Japanese): {translated_text}
Back-Translation (English): {back_translated_text}

Does the back-translation preserve the original meaning? Are there any changes in tone, information, or accuracy? Please point out specific issues and rate the translation on a scale from 1 (poor) to 5 (excellent)."""


In [ ]:
def evaluate_with_llm_with_back_translation(sample_data):
    for item in sample_data:
        source = item["source_text"] # English
        translation = item["translated_text"] # Japenese

        # Step 1: Get back-translation
        back_translated = get_back_translation(translation) # English

        # Step 2: Build judge prompt with back-translation included
        prompt = build_judge_prompt_with_back_translation(source, translation, back_translated)

        print("🔍 Evaluating:")
        print(f"Original: {source}")
        print(f"Translation: {translation}")
        print(f"Back-Translation: {back_translated}")
        print("🧠 GPT Response:")


        # Step 3: LLM as judge
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a strict bilingual translation evaluator. Use the back-translation to catch subtle errors."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        print(response.choices[0].message.content)

# Example usage:
evaluate_with_llm_with_back_translation(sample_data)


🔍 Evaluating:
Original: We need to finalize the contract before the end of the month.
Translation: 月末までに契約を確定させる必要があります。
Back-Translation: We need to finalize the contract by the end of the month.
🧠 GPT Response:
The back-translation perfectly preserves the original meaning. There are no changes in tone, information, or accuracy. The translation is accurate and the message is conveyed correctly. Therefore, I would rate this translation as 5 (excellent).
🔍 Evaluating:
Original: Schedule a meeting with the design team for next Tuesday.
Translation: 来週の日曜日にデザインチームとの会議を予定してください。
Back-Translation: Please schedule a meeting with the design team for next Sunday.
🧠 GPT Response:
The back-translation does not preserve the original meaning. The specific issue is the day of the week: the original text specifies the meeting for "next Tuesday," but the translation and back-translation refer to "next Sunday." This is a significant error in accuracy. The tone and information, aside from the day of th